In [2]:
from gsloc.inference.test import TestConfig, Test
from pathlib import Path
from gsloc.models import opr_graph_extention as network 
import torch
from torchvision.transforms import functional as F
from mmpr.models import MegaLoc
from gsloc.datasets import ThreeRScan

from torchvision import transforms as T
from gsloc.utils.visual import plot_metrics_from_parquet, plot_metrics_from_experiment_dir
from gsloc.models import FoLBase

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


2026-05-28 09:57:30.083 | WARNING  | opr.optional_deps:warn_once:115 - MinkowskiEngine is not available. sparse convolutions will be disabled. See the documentation for installation instructions


In [2]:
import random
import numpy as np

def make_deterministic(seed=0):
    """Make results deterministic. If seed == -1, do not make deterministic.
    Running the script in a deterministic way might slow it down.
    """
    if seed == -1:
        return
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
make_deterministic(0)

In [3]:
similarity_kwargs_list = [
    {
        "mode": "room",
        "trans_tol_m": 3,
        "rot_tol_deg": 180
    },
    {
        "mode": "pose",
        "trans_tol_m": 3,
        "rot_tol_deg": 180
    },
    {
        "mode": "pose",
        "trans_tol_m": 2,
        "rot_tol_deg": 90
    },
]

seq_filter_kwargs_list = [
    {
        "seq_similarity_filter_mode": "none",
        "seq_similarity_trans_tol_m": 0.5,
        "seq_similarity_rot_tol_deg": 15
    },
    {
        "seq_similarity_filter_mode": "pose",
        "seq_similarity_trans_tol_m": 0.5,
        "seq_similarity_rot_tol_deg": 15
    },
    {
        "seq_similarity_filter_mode": "pose",
        "seq_similarity_trans_tol_m": 1,
        "seq_similarity_rot_tol_deg": 30
    },
]

image_transform_fn = T.Compose([
    T.ToTensor(),
    T.Resize([322, 322], antialias=True),
    T.Lambda(lambda x: F.rotate(x, angle=-90)),  # 90° clockwise
    T.Normalize(
        mean=[0.44420420130352495, 0.41322746532289134, 0.3678658064565412], 
        std=[0.24352604373543688, 0.24045797651069503, 0.24250136992133814]
    ),
])

In [4]:
weights_path = Path("/mnt/external_usb_hdd/6YL/Datasets/3RScan/weights/graphs/gatv1/best_model.pth")

ckpt = torch.load(weights_path, map_location="cpu", weights_only=False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

OPR_GAT_graph_encoder = network.OPR_GATGraphEncoder(
    in_dim=4,
    hidden_dim=512,
    n_layers=1,
    num_node_classes=529, 
    node_emb_dim=128,
    num_edge_classes=41,
    edge_emb_dim=128,
    proj_dim=256,
    edge_cont_dim=10,
    dropout=0.1,
    heads=4
    ).to(device)
    
# megaloc = torch.hub.load("gmberton/MegaLoc", "get_trained_model")
# image_encoder = megaloc.to(device)

graph_model256 = network.OPR_MultiModalVPRGraphEncoder(
    graph_encoder=OPR_GAT_graph_encoder,
    image_encoder=None,
    image_out_dim=8448,
    graph_out_dim=256,
    fusion_dim=8448,
    normalize=True,
    graph_fusion_scale=0.05,
    freeze_image_encoder=True,
    mode="graph")

missing, unexpected = graph_model256.load_state_dict(ckpt["model_state_dict"], strict=False)
ignored_unexpected_prefixes = ("image_encoder.", "graph_encoder.convs.")
unexpected_other = [k for k in unexpected if not k.startswith(ignored_unexpected_prefixes)]
if unexpected_other:
    raise RuntimeError(f"Unexpected checkpoint keys: {unexpected_other}")
# ``missing`` includes MegaLoc hub weights and GINE conv params; those ckpt tensors appear under ``ignored_unexpected_prefixes``.

graph_model256.to(device)
graph_model256.eval()

OPR_MultiModalVPRGraphEncoder(
  (graph_encoder): OPR_GATGraphEncoder(
    (edge_cont_ln): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (edge_lbl_ln): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (node_emb): Embedding(529, 128)
    (edge_emb): Embedding(41, 128)
    (edge_cont_mlp): Sequential(
      (0): Linear(in_features=10, out_features=512, bias=True)
      (1): ReLU(inplace=True)
      (2): Linear(in_features=512, out_features=512, bias=True)
    )
    (edge_gate): Sequential(
      (0): Linear(in_features=1024, out_features=512, bias=True)
      (1): ReLU(inplace=True)
      (2): Linear(in_features=512, out_features=512, bias=True)
      (3): Sigmoid()
    )
    (edge_label_proj): Sequential(
      (0): Linear(in_features=128, out_features=512, bias=True)
      (1): ReLU(inplace=True)
      (2): Linear(in_features=512, out_features=512, bias=True)
    )
    (edge_fuse): Sequential(
      (0): Linear(in_features=1024, out_features=512, bias=True)
      (

In [5]:
weights_path = Path("/mnt/external_usb_hdd/6YL/Datasets/3RScan/weights/graphs/gatV3/best_model.pth")
ckpt = torch.load(weights_path, map_location="cpu", weights_only=False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

GAT_graph_encoder = network.OPR_GATGraphEncoder64(
    in_dim=4,
    hidden_dim=512,
    n_layers=1,
    num_node_classes=529, 
    node_emb_dim=64,
    num_edge_classes=41,
    edge_emb_dim=128,
    proj_dim=64,
    edge_cont_dim=10,
    dropout=0.1,
    heads=4).to(device)

graph_model64 = network.OPR_GraphEnhancedMegaloc64(
    graph_encoder=GAT_graph_encoder,
    image_encoder=None
)

missing, unexpected = graph_model64.load_state_dict(ckpt["multimodal_state_dict"], strict=False)
ignored_unexpected_prefixes = ("image_encoder.", "graph_encoder.convs.")
unexpected_other = [k for k in unexpected if not k.startswith(ignored_unexpected_prefixes)]
if unexpected_other:
    raise RuntimeError(f"Unexpected checkpoint keys: {unexpected_other}")
# ``missing`` includes MegaLoc hub weights and GINE conv params; those ckpt tensors appear under ``ignored_unexpected_prefixes``.

graph_model64.to(device)
graph_model64.eval()

OPR_GraphEnhancedMegaloc64(
  (graph_encoder): OPR_GATGraphEncoder64(
    (edge_cont_ln): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (edge_lbl_ln): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (node_emb): Embedding(529, 64)
    (edge_emb): Embedding(41, 128)
    (edge_cont_mlp): Sequential(
      (0): Linear(in_features=10, out_features=512, bias=True)
      (1): ReLU(inplace=True)
      (2): Linear(in_features=512, out_features=512, bias=True)
    )
    (edge_gate): Sequential(
      (0): Linear(in_features=1024, out_features=512, bias=True)
      (1): ReLU(inplace=True)
      (2): Linear(in_features=512, out_features=512, bias=True)
      (3): Sigmoid()
    )
    (edge_label_proj): Sequential(
      (0): Linear(in_features=128, out_features=512, bias=True)
      (1): ReLU(inplace=True)
      (2): Linear(in_features=512, out_features=512, bias=True)
    )
    (edge_fuse): Sequential(
      (0): Linear(in_features=1024, out_features=512, bias=True)
      (1)

In [6]:
fol_base = FoLBase()  # веса из weights/FoL_base.pth
fol_base = fol_base.to(device)
fol_base.eval()

Using cache found in /home/kartashov_ga/.cache/torch/hub/facebookresearch_dinov2_main
/home/kartashov_ga/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/home/kartashov_ga/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/home/kartashov_ga/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


FoLBase(
  (model): FoLNet(
    (backbone): DINOv2(
      (model): DinoVisionTransformer(
        (patch_embed): PatchEmbed(
          (proj): Conv2d(3, 768, kernel_size=(14, 14), stride=(14, 14))
          (norm): Identity()
        )
        (blocks): ModuleList(
          (0-11): 12 x NestedTensorBlock(
            (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
            (attn): MemEffAttention(
              (qkv): Linear(in_features=768, out_features=2304, bias=True)
              (proj): Linear(in_features=768, out_features=768, bias=True)
              (proj_drop): Dropout(p=0.0, inplace=False)
            )
            (ls1): LayerScale()
            (drop_path1): Identity()
            (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
            (mlp): Mlp(
              (fc1): Linear(in_features=768, out_features=3072, bias=True)
              (act): GELU(approximate='none')
              (fc2): Linear(in_features=3072, out_features=768, bias=T

In [7]:
megaLoc = MegaLoc()
megaLoc.to(device)
megaLoc.eval()

Using cache found in /home/kartashov_ga/.cache/torch/hub/gmberton_MegaLoc_main
Using cache found in /home/kartashov_ga/.cache/torch/hub/facebookresearch_dinov2_main


MegaLoc(
  (model): MegaLocModel(
    (backbone): DINOv2(
      (model): DinoVisionTransformer(
        (patch_embed): PatchEmbed(
          (proj): Conv2d(3, 768, kernel_size=(14, 14), stride=(14, 14))
          (norm): Identity()
        )
        (blocks): ModuleList(
          (0-11): 12 x NestedTensorBlock(
            (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
            (attn): MemEffAttention(
              (qkv): Linear(in_features=768, out_features=2304, bias=True)
              (proj): Linear(in_features=768, out_features=768, bias=True)
              (proj_drop): Dropout(p=0.0, inplace=False)
            )
            (ls1): LayerScale()
            (drop_path1): Identity()
            (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
            (mlp): Mlp(
              (fc1): Linear(in_features=768, out_features=3072, bias=True)
              (act): GELU(approximate='none')
              (fc2): Linear(in_features=3072, out_features=768, 

In [8]:
def run_test(
    tests_path, 
    dataset_path, 
    dataset_name,
    date, graph_type, 
    graphmodel_type, 
    image_model_type, 
    rerank_k, 
    per_frame_k, 
    filter_type, 
    similarity_type, 
    graph_dir, 
    edge_normalizer_path, 
    scene_list_path, 
    query_list_path,
    room_json_path,
    seq_filter_kwargs,
    similarity_kwargs,
    graph_model, 
    image_model,
    time_test
    ):

    model_name = f"{graphmodel_type}x{image_model_type}" if image_model_type != "None" and graphmodel_type != "None" \
        else (graphmodel_type + "_pure") if graphmodel_type != "None" else image_model_type
    today_dataset_test_path = tests_path / date / dataset_name
    test_path = today_dataset_test_path / graph_type / model_name if graph_type != "None" else today_dataset_test_path / model_name
    index_path = today_dataset_test_path / "cache" / "indexes" / graph_type / (graphmodel_type + "graph") if graph_type != "None" else today_dataset_test_path / "cache" / "indexes" / image_model_type
    query_cache_path = today_dataset_test_path / "cache" / "query_cache" / graph_type / (graphmodel_type + "graph") if graph_type != "None" else today_dataset_test_path / "cache" / "query_cache" / image_model_type
    rerank_index_path = today_dataset_test_path / "cache" / "indexes" / image_model_type if graph_type != "None" else "None"
    rerank_query_cache_path = today_dataset_test_path / "cache" / "query_cache" / image_model_type if graph_type != "None" else "None"
    frames_path = test_path / ("rerank_k_" + str(rerank_k) + "_per_frame_k_" + str(per_frame_k)) / "frames.npz"
    bench_report_path = test_path / ("rerank_k_" + str(rerank_k) + "_per_frame_k_" + str(per_frame_k)) / filter_type / similarity_type


    cfg = TestConfig(
        dataset_path=dataset_path,
        test_path=test_path,
        index_path=index_path,
        rerank_index_path=rerank_index_path,
        query_cache_path=query_cache_path,
        rerank_query_cache_path=rerank_query_cache_path,
        bench_report_path=bench_report_path,
        graph_path=graph_dir,   
        dataset_class=ThreeRScan,
        filter_kwargs={"similarity_filter_mode": "none"},
        seq_filter_kwargs=seq_filter_kwargs,
        scene_list_path=scene_list_path,
        query_list_path=query_list_path,
        room_json_path=room_json_path,
        edge_normalizer_path=edge_normalizer_path,
        image_transform_fn=image_transform_fn,
        graph_feat_dim=4,
        graph_edge_attr_dim=10,
        graph_rotate=True,
        device=device,
        batch_size=16,
        time_test=time_test,
        num_workers=4,
        model=graph_model if graph_model is not None else image_model,
        rerank_model=image_model if graph_model is not None else None,
        rerank_k=rerank_k,
        per_frame_k_used=per_frame_k,
        final_k=25,
        seq_lengths=[1, 2, 3, 5, 7, 10, 15, 20, 25, 30, 35],
        recall_at_k=[1, 5, 10, 25],
        similarity_kwargs=similarity_kwargs,
        std_mode="global",
        scene_df_field="scene",
        pose_df_field="pose",
        frames_path=frames_path
    )

    test = Test(cfg)
    test.run()

In [9]:
tests_path = Path("/home/kartashov_ga/projects/GSLoc/data/tests")
dataset_path = Path("/mnt/external_usb_hdd/6YL/Datasets/3RScan")
date = "26-05-26"
dataset_name = "3RScan" #"3RScan_BIG"
graph_type = "None" #"GT"
graphmodel_type = "None"#"64"
graph_model = None#graph_model64
image_model_type = "Megaloc"
image_model = megaLoc

rerank_k_list = [50, 100, 250, 500, 750,1000]
rerank_k = rerank_k_list[3]

per_frame_k_list = [10, 25, 50, 100]
per_frame_k = per_frame_k_list[1]

filter_names = ["base_seq_report", "nearfilter_seq_report", "farfilter_seq_report"]
filter_type = filter_names[0]
seq_filter_kwargs = seq_filter_kwargs_list[0]

similarity_names = ["room-sim", "pose-far-sim", "pose-near-sim"]
similarity_type = similarity_names[1]

graph_dir = "SceneGraphs_real_classes_pt_compact"
edge_normalizer_path="/mnt/external_usb_hdd/6YL/Datasets/3RScan/weights/graphs/gatv1/edge_normalizer (1).pt"
scene_list_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/test_resplit_scans.txt"# "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/BIG_test_scans_db.txt"
query_list_path = None#/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/BIG_test_scans_q.txt"
room_json_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/3RScan.json"

time_test = True

In [10]:
for i in range(len(similarity_kwargs_list)):
        run_test(
            tests_path=tests_path, 
            dataset_path=dataset_path, 
            dataset_name=dataset_name,
            date=date, 
            graph_type=graph_type, 
            graphmodel_type=graphmodel_type, 
            image_model_type=image_model_type, 
            rerank_k=rerank_k_list[3], 
            per_frame_k=per_frame_k, 
            filter_type=filter_type, 
            similarity_type=similarity_names[i], 
            graph_dir=graph_dir, 
            edge_normalizer_path=edge_normalizer_path, 
            scene_list_path=scene_list_path, 
            query_list_path=query_list_path,
            room_json_path=room_json_path, 
            seq_filter_kwargs=seq_filter_kwargs, 
            similarity_kwargs=similarity_kwargs_list[i],
            graph_model=graph_model,
            image_model=image_model, 
            time_test=time_test
            )

2026-05-26 16:12:13.717 | INFO     | mmpr.inference.index:generate:445 - Using existing meta.parquet
2026-05-26 16:12:13.717 | INFO     | mmpr.inference.index:generate:474 - Using existing descriptors.npy


2026-05-26 16:12:13.794 | INFO     | mmpr.inference.index:generate:493 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-05-26/3RScan/cache/indexes/Megaloc


Loading frames from /home/kartashov_ga/projects/GSLoc/data/tests/26-05-26/3RScan/Megaloc/rerank_k_500_per_frame_k_25/frames.npz


  0%|          | 0/11 [00:00<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
tests_path = Path("/home/kartashov_ga/projects/GSLoc/data/tests")
dataset_path = Path("/mnt/external_usb_hdd/6YL/Datasets/3RScan")
date = "26-05-19"
dataset_name = "3RScan" #"3RScan_BIG"
graph_type = "None" #"GT"
graphmodel_type = "None"#"64"
graph_model = None#graph_model64
image_model_type = "Fol_base"
image_model = fol_base

rerank_k_list = [50, 100, 250, 500, 750,1000]
rerank_k = rerank_k_list[3]

per_frame_k_list = [10, 25, 50, 100]
per_frame_k = per_frame_k_list[1]

filter_names = ["base_seq_report", "nearfilter_seq_report", "farfilter_seq_report"]
filter_type = filter_names[0]
seq_filter_kwargs = seq_filter_kwargs_list[0]

similarity_names = ["room-sim", "pose-far-sim", "pose-near-sim"]
similarity_type = similarity_names[1]

graph_dir = "SceneGraphs_real_classes_pt_compact"
edge_normalizer_path="/mnt/external_usb_hdd/6YL/Datasets/3RScan/weights/graphs/gatv1/edge_normalizer (1).pt"
scene_list_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/test_resplit_scans.txt"# "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/BIG_test_scans_db.txt"
query_list_path = None#/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/BIG_test_scans_q.txt"
room_json_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/3RScan.json"

In [ ]:
for i in range(len(similarity_kwargs_list)):
        run_test(
            tests_path=tests_path, 
            dataset_path=dataset_path, 
            dataset_name=dataset_name,
            date=date, 
            graph_type=graph_type, 
            graphmodel_type=graphmodel_type, 
            image_model_type=image_model_type, 
            rerank_k=rerank_k_list[3], 
            per_frame_k=per_frame_k, 
            filter_type=filter_type, 
            similarity_type=similarity_names[i], 
            graph_dir=graph_dir, 
            edge_normalizer_path=edge_normalizer_path, 
            scene_list_path=scene_list_path, 
            query_list_path=query_list_path,
            room_json_path=room_json_path, 
            seq_filter_kwargs=seq_filter_kwargs, 
            similarity_kwargs=similarity_kwargs_list[i],
            graph_model=graph_model,
            image_model=image_model,   
            )

2026-05-20 22:18:41.654 | INFO     | gsloc.datasets.three_rscan:__init__:353 - Metadata not found, rebuilding metadata for 3rscan dataset
2026-05-20 22:18:41.655 | INFO     | gsloc.datasets.three_rscan:build_3rscan_df:210 - Scanning 3rscan dataset for 93 selected scenes...


2026-05-20 22:18:53.601 | INFO     | gsloc.datasets.three_rscan:build_3rscan_df:288 - Scanned 21013 rows
2026-05-20 22:18:53.606 | INFO     | mmpr.inference.index:generate:439 - Using existing meta.parquet
100%|██████████| 591/591 [02:11<00:00,  4.49it/s]
2026-05-20 22:21:05.556 | INFO     | mmpr.inference.index:generate:466 - descriptors.npy file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-05-19/3RScan/cache/indexes/Fol_base
2026-05-20 22:21:05.716 | INFO     | mmpr.inference.index:generate:486 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-05-19/3RScan/cache/indexes/Fol_base


Running inference and saving frames to /home/kartashov_ga/projects/GSLoc/data/tests/26-05-19/3RScan/Fol_base/rerank_k_500_per_frame_k_25/frames.npz


Compute descriptors + PR cache: 100%|██████████| 1314/1314 [07:03<00:00,  3.10it/s]
2026-05-20 22:28:10.028 | INFO     | gsloc.datasets.pr_dataset:save_meta_parquet:51 - Wrote 21,013 rows to /home/kartashov_ga/projects/GSLoc/data/tests/26-05-19/3RScan/cache/query_cache/Fol_base/meta.parquet
100%|██████████| 11/11 [04:46<00:00, 26.04s/it]
2026-05-20 22:32:56.995 | INFO     | mmpr.inference.index:generate:439 - Using existing meta.parquet
2026-05-20 22:32:56.995 | INFO     | mmpr.inference.index:generate:468 - Using existing descriptors.npy
2026-05-20 22:32:57.053 | INFO     | mmpr.inference.index:generate:486 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-05-19/3RScan/cache/indexes/Fol_base


Index search time mean: 0.09578471845841156
Loading frames from /home/kartashov_ga/projects/GSLoc/data/tests/26-05-19/3RScan/Fol_base/rerank_k_500_per_frame_k_25/frames.npz


  0%|          | 0/11 [00:00<?, ?it/s]/home/kartashov_ga/projects/GSLoc/src/gsloc/datasets/three_rscan.py:625: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  pose_a = torch.as_tensor(pose_a)
100%|██████████| 11/11 [10:15<00:00, 55.92s/it]
2026-05-20 22:43:12.548 | INFO     | mmpr.inference.index:generate:439 - Using existing meta.parquet
2026-05-20 22:43:12.549 | INFO     | mmpr.inference.index:generate:468 - Using existing descriptors.npy
2026-05-20 22:43:12.607 | INFO     | mmpr.inference.index:generate:486 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-05-19/3RScan/cache/indexes/Fo

Index search time mean: 0.0
Loading frames from /home/kartashov_ga/projects/GSLoc/data/tests/26-05-19/3RScan/Fol_base/rerank_k_500_per_frame_k_25/frames.npz


100%|██████████| 11/11 [09:37<00:00, 52.53s/it]

Index search time mean: 0.0


In [8]:
tests_path = Path("/home/kartashov_ga/projects/GSLoc/data/tests")
dataset_path = Path("/mnt/external_usb_hdd/6YL/Datasets/3RScan")
date = "26-05-19"
dataset_name = "3RScan" #"3RScan_BIG"
graph_type = "GT"
graphmodel_type = "64"
graph_model = graph_model64
image_model_type = "None"
image_model = None

rerank_k_list = [50, 100, 250, 500, 750,1000]
rerank_k = rerank_k_list[3]

per_frame_k_list = [10, 25, 50, 100]
per_frame_k = per_frame_k_list[1]

filter_names = ["base_seq_report", "nearfilter_seq_report", "farfilter_seq_report"]
filter_type = filter_names[0]
seq_filter_kwargs = seq_filter_kwargs_list[0]

similarity_names = ["room-sim", "pose-far-sim", "pose-near-sim"]
similarity_type = similarity_names[1]

graph_dir = "SceneGraphs_real_classes_pt_compact"
edge_normalizer_path="/mnt/external_usb_hdd/6YL/Datasets/3RScan/weights/graphs/gatv1/edge_normalizer (1).pt"
scene_list_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/test_resplit_scans.txt"# "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/BIG_test_scans_db.txt"
query_list_path = None#/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/BIG_test_scans_q.txt"
room_json_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/3RScan.json"

In [9]:
for i in range(len(similarity_kwargs_list)):
        run_test(
            tests_path=tests_path, 
            dataset_path=dataset_path, 
            dataset_name=dataset_name,
            date=date, 
            graph_type=graph_type, 
            graphmodel_type=graphmodel_type, 
            image_model_type=image_model_type, 
            rerank_k=rerank_k_list[3], 
            per_frame_k=per_frame_k, 
            filter_type=filter_type, 
            similarity_type=similarity_names[i], 
            graph_dir=graph_dir, 
            edge_normalizer_path=edge_normalizer_path, 
            scene_list_path=scene_list_path, 
            query_list_path=query_list_path,
            room_json_path=room_json_path, 
            seq_filter_kwargs=seq_filter_kwargs, 
            similarity_kwargs=similarity_kwargs_list[i],
            graph_model=graph_model,
            image_model=image_model,   
            )

TypeError: run_test() missing 1 required positional argument: 'time_test'

In [11]:
tests_path = Path("/home/kartashov_ga/projects/GSLoc/data/tests")
dataset_path = Path("/mnt/external_usb_hdd/6YL/Datasets/3RScan")
date = "26-05-26"
dataset_name = "3RScan" #"3RScan_BIG"
graph_type = "GT"
graphmodel_type = "64"
graph_model = graph_model64
image_model_type = "Megaloc"
image_model = megaLoc

rerank_k_list = [1000, 2000]
rerank_k = rerank_k_list[0]

per_frame_k_list = [10, 25, 50, 100]
per_frame_k = per_frame_k_list[1]

filter_names = ["base_seq_report", "nearfilter_seq_report", "farfilter_seq_report"]
filter_type = filter_names[0]
seq_filter_kwargs = seq_filter_kwargs_list[0]

similarity_names = ["room-sim", "pose-far-sim", "pose-near-sim"]
similarity_type = similarity_names[1]

graph_dir = "SceneGraphs_real_classes_pt_compact"
edge_normalizer_path="/mnt/external_usb_hdd/6YL/Datasets/3RScan/weights/graphs/gatv1/edge_normalizer (1).pt"
scene_list_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/test_resplit_scans.txt"# "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/BIG_test_scans_db.txt"
query_list_path = None#/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/BIG_test_scans_q.txt"
room_json_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/3RScan.json"

time_test = True

In [12]:
#for j in range(len(seq_filter_kwargs_list)):
for j in range(len(rerank_k_list)):
    for i in range(len(similarity_kwargs_list)):
        run_test(
            tests_path=tests_path, 
            dataset_path=dataset_path, 
            dataset_name=dataset_name,
            date=date, 
            graph_type=graph_type, 
            graphmodel_type=graphmodel_type, 
            image_model_type=image_model_type, 
            rerank_k=rerank_k_list[j], 
            per_frame_k=per_frame_k, 
            filter_type=filter_type, 
            similarity_type=similarity_names[i], 
            graph_dir=graph_dir, 
            edge_normalizer_path=edge_normalizer_path, 
            scene_list_path=scene_list_path, 
            query_list_path=query_list_path,
            room_json_path=room_json_path, 
            seq_filter_kwargs=seq_filter_kwargs, 
            similarity_kwargs=similarity_kwargs_list[i],
            graph_model=graph_model,
            image_model=image_model,   
            time_test=time_test
            )

2026-05-26 16:13:05.666 | INFO     | mmpr.inference.index:generate:445 - Using existing meta.parquet
2026-05-26 16:13:05.667 | INFO     | mmpr.inference.index:generate:474 - Using existing descriptors.npy
2026-05-26 16:13:05.669 | INFO     | mmpr.inference.index:generate:493 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-05-26/3RScan/cache/indexes/GT/64graph
2026-05-26 16:13:05.710 | INFO     | mmpr.inference.index:generate:445 - Using existing meta.parquet
2026-05-26 16:13:05.710 | INFO     | mmpr.inference.index:generate:474 - Using existing descriptors.npy


2026-05-26 16:13:05.772 | INFO     | mmpr.inference.index:generate:493 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-05-26/3RScan/cache/indexes/Megaloc


Running inference and saving frames to /home/kartashov_ga/projects/GSLoc/data/tests/26-05-26/3RScan/GT/64xMegaloc/rerank_k_1000_per_frame_k_25/frames.npz


Time test: 100%|██████████| 100/100 [00:04<00:00, 24.06it/s]


Using cached descriptors for query and rerank: loading from  /home/kartashov_ga/projects/GSLoc/data/tests/26-05-26/3RScan/cache/query_cache/GT/64graph /home/kartashov_ga/projects/GSLoc/data/tests/26-05-26/3RScan/cache/query_cache/Megaloc
Descriptors loaded:  (21013, 64) (21013, 8448) getting results


...

Index1 search time mean: 0.00041695043200860475
Rerank index2 search time mean: 0.012714838700498759
Loading frames from /home/kartashov_ga/projects/GSLoc/data/tests/26-05-26/3RScan/GT/64xMegaloc/rerank_k_2000_per_frame_k_25/frames.npz


100%|██████████| 11/11 [10:57<00:00, 59.80s/it]
2026-05-26 17:03:15.530 | INFO     | mmpr.inference.index:generate:445 - Using existing meta.parquet
2026-05-26 17:03:15.531 | INFO     | mmpr.inference.index:generate:474 - Using existing descriptors.npy
2026-05-26 17:03:15.532 | INFO     | mmpr.inference.index:generate:493 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-05-26/3RScan/cache/indexes/GT/64graph
2026-05-26 17:03:15.558 | INFO     | mmpr.inference.index:generate:445 - Using existing meta.parquet
2026-05-26 17:03:15.559 | INFO     | mmpr.inference.index:generate:474 - Using existing descriptors.npy
2026-05-26 17:03:15.606 | INFO     | mmpr.inference.index:generate:493 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-05-26/3RScan/cache/indexes/Megaloc


Index1 search time mean: 0.0
Rerank index2 search time mean: 0.0
Loading frames from /home/kartashov_ga/projects/GSLoc/data/tests/26-05-26/3RScan/GT/64xMegaloc/rerank_k_2000_per_frame_k_25/frames.npz


100%|██████████| 11/11 [10:14<00:00, 55.83s/it]

Index1 search time mean: 0.0
Rerank index2 search time mean: 0.0


In [13]:
tests_path = Path("/home/kartashov_ga/projects/GSLoc/data/tests")
dataset_path = Path("/mnt/external_usb_hdd/6YL/Datasets/3RScan")
date = "26-05-26"
dataset_name = "3RScan" #"3RScan_BIG"
graph_type = "Makarov"
graphmodel_type = "64"
graph_model = graph_model64
image_model_type = "Megaloc"
image_model = megaLoc

rerank_k_list = [1000, 2000]
rerank_k = rerank_k_list[0]

per_frame_k_list = [10, 25, 50, 100]
per_frame_k = per_frame_k_list[1]

filter_names = ["base_seq_report", "nearfilter_seq_report", "farfilter_seq_report"]
filter_type = filter_names[0]
seq_filter_kwargs = seq_filter_kwargs_list[0]

similarity_names = ["room-sim", "pose-far-sim", "pose-near-sim"]
similarity_type = similarity_names[1]

graph_dir = "SceneGraphs_Makarov_FULL_TEST_pt"
edge_normalizer_path="/mnt/external_usb_hdd/6YL/Datasets/3RScan/weights/graphs/gatv1/edge_normalizer (1).pt"
scene_list_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/test_resplit_scans.txt"# "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/BIG_test_scans_db.txt"
query_list_path = None#/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/BIG_test_scans_q.txt"
room_json_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/3RScan.json"

time_test = True

In [14]:
# #for j in range(len(seq_filter_kwargs_list)):
# for j in range(len(rerank_k_list)):
for i in range(len(similarity_kwargs_list)):
    run_test(
        tests_path=tests_path, 
        dataset_path=dataset_path, 
        dataset_name=dataset_name,
        date=date, 
        graph_type=graph_type, 
        graphmodel_type=graphmodel_type, 
        image_model_type=image_model_type, 
        rerank_k=rerank_k_list[0], 
        per_frame_k=per_frame_k, 
        filter_type=filter_type, 
        similarity_type=similarity_names[i], 
        graph_dir=graph_dir, 
        edge_normalizer_path=edge_normalizer_path, 
        scene_list_path=scene_list_path, 
        query_list_path=query_list_path,
        room_json_path=room_json_path, 
        seq_filter_kwargs=seq_filter_kwargs, 
        similarity_kwargs=similarity_kwargs_list[i],
        graph_model=graph_model,
        image_model=image_model,   
        time_test=time_test
        )

2026-05-26 17:19:55.002 | INFO     | gsloc.datasets.three_rscan:__init__:353 - Metadata not found, rebuilding metadata for 3rscan dataset
2026-05-26 17:19:55.046 | INFO     | gsloc.datasets.three_rscan:build_3rscan_df:210 - Scanning 3rscan dataset for 30 selected scenes...
2026-05-26 17:20:04.938 | INFO     | gsloc.datasets.three_rscan:build_3rscan_df:288 - Scanned 9449 rows
2026-05-26 17:20:04.966 | INFO     | gsloc.datasets.three_rscan:__init__:353 - Metadata not found, rebuilding metadata for 3rscan dataset
2026-05-26 17:20:04.966 | INFO     | gsloc.datasets.three_rscan:build_3rscan_df:210 - Scanning 3rscan dataset for 93 selected scenes...
2026-05-26 17:20:17.012 | INFO     | gsloc.datasets.three_rscan:build_3rscan_df:288 - Scanned 21013 rows
2026-05-26 17:20:17.041 | INFO     | gsloc.datasets.pr_dataset:save_meta_parquet:51 - Wrote 9,449 rows to /home/kartashov_ga/projects/GSLoc/data/tests/26-05-26/3RScan/cache/indexes/Makarov/64graph/meta.parquet
2026-05-26 17:20:17.041 | INFO   

Running inference and saving frames to /home/kartashov_ga/projects/GSLoc/data/tests/26-05-26/3RScan/Makarov/64xMegaloc/rerank_k_1000_per_frame_k_25/frames.npz


Compute descriptors + PR cache:  30%|███       | 399/1314 [02:13<05:01,  3.03it/s]2026-05-26 17:23:14.120 | WARNING  | gsloc.datasets.three_rscan:_warn_missing_asset:451 - Missing or unreadable graph /mnt/external_usb_hdd/6YL/Datasets/3RScan/SceneGraphs_Makarov_FULL_TEST_pt/42384908-60a7-271e-9c46-01e562c8974c/frame-000017.pt
2026-05-26 17:23:14.132 | WARNING  | gsloc.datasets.three_rscan:_warn_missing_asset:451 - Missing or unreadable graph /mnt/external_usb_hdd/6YL/Datasets/3RScan/SceneGraphs_Makarov_FULL_TEST_pt/42384908-60a7-271e-9c46-01e562c8974c/frame-000018.pt
Compute descriptors + PR cache: 100%|██████████| 1314/1314 [07:15<00:00,  3.02it/s]
2026-05-26 17:28:16.118 | INFO     | gsloc.datasets.pr_dataset:save_meta_parquet:51 - Wrote 21,013 rows to /home/kartashov_ga/projects/GSLoc/data/tests/26-05-26/3RScan/cache/query_cache/Makarov/64graph/meta.parquet
2026-05-26 17:28:16.556 | INFO     | gsloc.datasets.pr_dataset:save_meta_parquet:51 - Wrote 21,013 rows to /home/kartashov_ga/p

Index1 search time mean: 0.0019185512544813005
Rerank index2 search time mean: 0.10098108008609379


2026-05-26 17:33:24.983 | INFO     | mmpr.inference.index:generate:445 - Using existing meta.parquet
2026-05-26 17:33:24.984 | INFO     | mmpr.inference.index:generate:474 - Using existing descriptors.npy
2026-05-26 17:33:24.985 | INFO     | mmpr.inference.index:generate:493 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-05-26/3RScan/cache/indexes/Makarov/64graph
2026-05-26 17:33:25.011 | INFO     | mmpr.inference.index:generate:445 - Using existing meta.parquet
2026-05-26 17:33:25.012 | INFO     | mmpr.inference.index:generate:474 - Using existing descriptors.npy
2026-05-26 17:33:25.068 | INFO     | mmpr.inference.index:generate:493 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-05-26/3RScan/cache/indexes/Megaloc


Loading frames from /home/kartashov_ga/projects/GSLoc/data/tests/26-05-26/3RScan/Makarov/64xMegaloc/rerank_k_1000_per_frame_k_25/frames.npz


100%|██████████| 11/11 [10:34<00:00, 57.70s/it]
2026-05-26 17:44:01.215 | INFO     | mmpr.inference.index:generate:445 - Using existing meta.parquet
2026-05-26 17:44:01.216 | INFO     | mmpr.inference.index:generate:474 - Using existing descriptors.npy
2026-05-26 17:44:01.218 | INFO     | mmpr.inference.index:generate:493 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-05-26/3RScan/cache/indexes/Makarov/64graph
2026-05-26 17:44:01.244 | INFO     | mmpr.inference.index:generate:445 - Using existing meta.parquet
2026-05-26 17:44:01.244 | INFO     | mmpr.inference.index:generate:474 - Using existing descriptors.npy
2026-05-26 17:44:01.300 | INFO     | mmpr.inference.index:generate:493 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-05-26/3RScan/cache/indexes/Megaloc


Index1 search time mean: 0.0
Rerank index2 search time mean: 0.0
Loading frames from /home/kartashov_ga/projects/GSLoc/data/tests/26-05-26/3RScan/Makarov/64xMegaloc/rerank_k_1000_per_frame_k_25/frames.npz


100%|██████████| 11/11 [09:53<00:00, 53.96s/it]

Index1 search time mean: 0.0
Rerank index2 search time mean: 0.0


In [ ]:
tests_path = Path("/home/kartashov_ga/projects/GSLoc/data/tests")
dataset_path = Path("/mnt/external_usb_hdd/6YL/Datasets/3RScan")
date = "26-05-26"
dataset_name = "3RScan" #"3RScan_BIG"
graph_type = "GT"
graphmodel_type = "64"
graph_model = graph_model64
image_model_type = "Megaloc"
image_model = megaLoc

rerank_k_list = [1000, 2000]
rerank_k = rerank_k_list[0]

per_frame_k_list = [10, 25, 50, 100]
per_frame_k = per_frame_k_list[1]

filter_names = ["base_seq_report", "nearfilter_seq_report", "farfilter_seq_report"]
filter_type = filter_names[0]
seq_filter_kwargs = seq_filter_kwargs_list[0]

similarity_names = ["room-sim", "pose-far-sim", "pose-near-sim"]
similarity_type = similarity_names[1]

graph_dir = "SceneGraphs_real_classes_pt_compact"
edge_normalizer_path="/mnt/external_usb_hdd/6YL/Datasets/3RScan/weights/graphs/gatv1/edge_normalizer (1).pt"
scene_list_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/test_resplit_scans.txt"# "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/BIG_test_scans_db.txt"
query_list_path = None#/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/BIG_test_scans_q.txt"
room_json_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/3RScan.json"

time_test = True

In [3]:
plot_metrics_from_parquet(
    "/home/kartashov_ga/projects/GSLoc/data/tests/26-05-26/3RScan/Makarov/64xMegaloc/rerank_k_1000_per_frame_k_25/base_seq_report/room-sim/summaryresults.parquet",
    "/home/kartashov_ga/projects/GSLoc/data/tests/26-05-19/3RScan/Megaloc/rerank_k_500_per_frame_k_25/base_seq_report/room-sim/summaryresults.parquet",
    )

{'auc_pr': Figure({
     'data': [{'hovertemplate': 'w=%{x}<br>auc_pr=%{y}<extra></extra>',
               'legendgroup': '',
               'line': {'color': '#636efa', 'dash': 'solid'},
               'marker': {'symbol': 'circle'},
               'mode': 'lines+markers',
               'name': '',
               'orientation': 'v',
               'showlegend': False,
               'type': 'scatter',
               'x': {'bdata': 'AQIDBQcKDxQZHiM=', 'dtype': 'i1'},
               'xaxis': 'x',
               'y': {'bdata': ('AAAAAAAAAAAAAAAAAAAAAAAAAAAAAA' ... 'AAAAAAAAAAAAAAAAAAAAAAAAAAAA=='),
                     'dtype': 'f8'},
               'yaxis': 'y'},
              {'marker': {'color': 'red', 'size': 10},
               'mode': 'markers',
               'name': 'max',
               'showlegend': False,
               'type': 'scatter',
               'x': [1],
               'y': [0.0]},
              {'line': {'color': 'purple', 'dash': 'dot'},
               'mode': 'lin